# Churn SaaS B2B - Analysis Notebook

## Objetivo
Estimar probabilidade de churn de contas SaaS para priorizar retencao.

## Perguntas de negocio
- Quais sinais estao mais ligados ao churn?
- Qual threshold gera melhor equilibrio entre recall e precision?
- Como transformar score em plano de acao?


### O que este código faz
Carrega bibliotecas, lê os arquivos principais do projeto (`data` e `metrics`) e mostra uma primeira amostra dos dados para conferência inicial.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path('..')
data_path = project_root / 'data' / 'churn_saas_synthetic.csv'
metrics_path = project_root / 'models' / 'metrics.json'

pd.set_option('display.max_columns', 200)
df = pd.read_csv(data_path)
metrics = json.loads(metrics_path.read_text(encoding='utf-8')) if metrics_path.exists() else {}

print('shape:', df.shape)
print('metrics keys:', list(metrics.keys()))
df.head(5)


### O que este código faz
Verifica qualidade básica dos dados: valores ausentes e distribuição da variável-alvo (quando existir). É um check rápido antes de qualquer análise.


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('missing columns:', len(missing))
if len(missing) > 0:
    display(missing)

if 'churn' in df.columns:
    print('target distribution:')
    print(df['churn'].value_counts(normalize=True).round(4))


### O que este código faz
Realiza uma análise exploratória específica do projeto para gerar visão inicial dos padrões principais.


In [ ]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()
num_cols = [c for c in num_cols if c != 'churn']

summary = df[num_cols].describe().T[['mean', 'std', 'min', 'max']]
summary.sort_values('std', ascending=False).head(15)


### O que este código faz
Gera visualizações complementares para facilitar interpretação e comunicação dos resultados.


In [ ]:
risk_features = ['days_since_last_login', 'support_tickets_last_90d', 'feature_adoption_score', 'late_payments_last_12m']
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), risk_features):
    ax.hist(df.loc[df['churn'] == 0, col], bins=30, alpha=0.7, label='no churn')
    ax.hist(df.loc[df['churn'] == 1, col], bins=30, alpha=0.7, label='churn')
    ax.set_title(col)
    ax.grid(alpha=0.2)
axes[0,0].legend()
plt.tight_layout()


## Performance do modelo
A tabela abaixo mostra as metricas finais do pipeline salvo em `models/metrics.json`.


### O que este código faz
Exibe as métricas salvas no pipeline para conectar análise exploratória com desempenho final do modelo.


In [ ]:
pd.DataFrame([metrics]).T.rename(columns={0: 'value'}).head(30)


## Status atual
- Este projeto ja foi executado de ponta a ponta.
- Artefatos (dados, modelos, metricas e relatorios) estao gerados.
- Repositorio publicado e sincronizado no GitHub.
